
# Solar Filament Segmentation Challenge 2026 — YOLO26 Segmentation Baseline

This notebook keeps the original baseline's overall structure:

- leakage-safe image-level train/validation split
- COCO annotations
- 512×512 training/inference size
- early stopping
- validation-based threshold selection
- connected-component cleanup
- native-resolution RLE submission
- training graphs and qualitative prediction review

The main model is changed from ResNet-34 U-Net to **Ultralytics YOLO26s instance segmentation**.

YOLO is trained through the Ultralytics API. Because the source annotations are COCO masks while YOLO expects polygon segmentation labels, the notebook creates a temporary YOLO-format dataset from the COCO annotations.

> Note: YOLO's pretrained pipeline uses its own image normalization and segmentation losses. The original U-Net BCE + Dice loss is therefore not reproduced.



## 1. Install dependencies and import libraries

YOLO26 is loaded from the current Ultralytics package. The `-seg` checkpoint is the instance-segmentation variant.


In [ ]:

%pip install -q -U ultralytics pycocotools scikit-learn pandas matplotlib tqdm

from pathlib import Path
import os
import random
import shutil
import time
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pycocotools.coco import COCO
from pycocotools import mask as mask_utils
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else "cpu"

print(f"Training device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA devices available: {torch.cuda.device_count()}")



## 2. Configure paths and experiment

The original experiment used a single GPU even when more than one GPU was visible, so this notebook also uses GPU 0 for a simple baseline.

The dataset builder uses symlinks where possible. This lets multiple COCO records for the same physical JPEG remain separate training samples without copying the large source image repeatedly.


In [ ]:

SEED = 42

IMAGE_SIZE = 512
BATCH_SIZE = 8 if DEVICE.type == "cuda" else 2
NUM_EPOCHS = 100
VALIDATION_FRACTION = 0.20
PATIENCE = 10

NUM_WORKERS = 4
MIN_COMPONENT_AREA = 20

YOLO_MODEL_NAME = "yolo26s-seg.pt"
YOLO_DATA_DIR = Path.cwd() / "yolo_filament_dataset"
RUNS_DIR = Path.cwd() / "yolo_runs"
RUN_NAME = "yolo26s_filament"

DATA_YAML = YOLO_DATA_DIR / "filament.yaml"
CHECKPOINT_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
SUBMISSION_PATH = Path.cwd() / "solar_filament_yolo26s_submission.csv"

CREATE_SUBMISSION = True
REBUILD_YOLO_DATASET = True

CONFIDENCE_CANDIDATES = (0.15, 0.25, 0.35, 0.45, 0.55)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    candidates = [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]

    for parent in candidates:
        if not parent.exists():
            continue

        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent

            if (
                (root / "train" / "train_images").is_dir()
                and (root / "test" / "test_images").is_dir()
            ):
                return root

    raise FileNotFoundError(
        "Could not find MAGFiLO_1.0_Kaggle_2026 "
        "with train and test image folders."
    )

DATA_ROOT = find_dataset_root()
TRAIN_DIR = DATA_ROOT / "train"
TRAIN_IMAGE_DIR = TRAIN_DIR / "train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test" / "test_images"
ANNOTATION_PATH = TRAIN_DIR / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

print(f"Dataset root: {DATA_ROOT}")
print(f"Annotation JSON: {ANNOTATION_PATH}")
print(f"YOLO model: {YOLO_MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")



## 3. Load annotations and create a leakage-safe split

The split is still performed by **physical filename**, not by COCO record ID. This keeps repeated annotator records for the same JPEG on the same side of the split.


In [ ]:

coco = COCO(str(ANNOTATION_PATH))

image_ids = list(coco.imgs.keys())

records = pd.DataFrame([
    {
        "image_id": image_id,
        "file_name": coco.imgs[image_id]["file_name"],
    }
    for image_id in image_ids
])

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(records, groups=records["file_name"])
)

train_records = records.iloc[train_idx].reset_index(drop=True)
val_records = records.iloc[val_idx].reset_index(drop=True)

train_ids = train_records["image_id"].tolist()
val_ids = val_records["image_id"].tolist()

train_files = set(train_records["file_name"])
val_files = set(val_records["file_name"])

assert train_files.isdisjoint(val_files), (
    "A physical image appears in both splits."
)

print(f"Training records: {len(train_records)} across {len(train_files)} images")
print(f"Validation records: {len(val_records)} across {len(val_files)} images")
print(f"Train/validation filename overlap: {len(train_files & val_files)}")



## 4. Convert COCO segmentation masks to YOLO segmentation labels

Ultralytics YOLO segmentation expects one line per instance:

`class_id x1 y1 x2 y2 ...`

where polygon coordinates are normalized to `[0, 1]`.

Each COCO record gets its own unique YOLO sample name. The image itself is symlinked to the original JPEG when the filesystem permits it, so duplicate annotator records do not require duplicate image storage.

The source JPEGs are grayscale, but OpenCV/Ultralytics loads them as three-channel BGR images for the pretrained YOLO model.


In [ ]:

def mask_to_yolo_segments(mask):
    mask = (mask > 0).astype(np.uint8)

    height, width = mask.shape

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask,
        connectivity=8,
    )

    segments = []

    for label in range(1, count):
        area = int(stats[label, cv2.CC_STAT_AREA])

        if area < 3:
            continue

        component = (labels == label).astype(np.uint8)

        contours, _ = cv2.findContours(
            component,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )

        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea)

        if len(contour) < 3 or cv2.contourArea(contour) <= 0:
            continue

        points = contour.reshape(-1, 2).astype(np.float32)

        points[:, 0] /= width
        points[:, 1] /= height

        points = np.clip(points, 0.0, 1.0)

        if len(points) >= 3:
            segments.append(points)

    return segments


def record_to_yolo_label_lines(image_id):
    info = coco.loadImgs([image_id])[0]
    annotations = coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))

    lines = []

    for annotation in annotations:
        mask = coco.annToMask(annotation).astype(np.uint8)

        for points in mask_to_yolo_segments(mask):
            flat = points.reshape(-1)

            values = ["0"]
            values.extend(f"{value:.6f}" for value in flat)

            lines.append(" ".join(values))

    return lines


def create_image_link(source_path, destination_path):
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if destination_path.exists() or destination_path.is_symlink():
        destination_path.unlink()

    try:
        destination_path.symlink_to(source_path.resolve())
    except OSError:
        shutil.copy2(source_path, destination_path)


def write_yolo_split(split_records, split_name):
    image_dir = YOLO_DATA_DIR / "images" / split_name
    label_dir = YOLO_DATA_DIR / "labels" / split_name

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    empty_labels = 0
    total_instances = 0

    progress = tqdm(
        split_records.itertuples(index=False),
        total=len(split_records),
        desc=f"Preparing YOLO {split_name}",
        unit="record",
    )

    for row in progress:
        source_path = TRAIN_IMAGE_DIR / row.file_name

        if not source_path.is_file():
            raise FileNotFoundError(
                f"Could not find source image: {source_path}"
            )

        source_stem = Path(row.file_name).stem
        sample_stem = f"{int(row.image_id):08d}_{source_stem}"

        image_path = image_dir / f"{sample_stem}.jpg"
        label_path = label_dir / f"{sample_stem}.txt"

        create_image_link(source_path, image_path)

        lines = record_to_yolo_label_lines(row.image_id)
        label_path.write_text(
            "\n".join(lines) + ("\n" if lines else ""),
            encoding="utf-8",
        )

        total_instances += len(lines)
        empty_labels += not bool(lines)

    print(
        f"{split_name}: {len(split_records)} records | "
        f"{total_instances:,} polygon instances | "
        f"empty labels: {empty_labels}"
    )


if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    shutil.rmtree(YOLO_DATA_DIR)

write_yolo_split(train_records, "train")
write_yolo_split(val_records, "val")

DATA_YAML.write_text(
    (
        f"path: '{YOLO_DATA_DIR.as_posix()}'\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: filament\n"
    ),
    encoding="utf-8",
)

print(f"YOLO dataset configuration: {DATA_YAML}")
print(DATA_YAML.read_text())



## 5. Build and train YOLO26s-seg

The augmentation settings are intentionally close to the original notebook:

- horizontal/vertical flips
- small rotations
- small translations and scaling
- mild intensity variation
- no mosaic/mixup/copy-paste

Mosaic-style composition is disabled because these are large solar images containing thin filament structures, and this baseline is intended to stay close to the original image-wise augmentation setup.


In [ ]:

print("Building pretrained YOLO26s segmentation model...")

model = YOLO(YOLO_MODEL_NAME)

training_start = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=YOLO_DEVICE,
    workers=NUM_WORKERS,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    pretrained=True,
    optimizer="auto",
    amp=DEVICE.type == "cuda",

    seed=SEED,
    deterministic=True,

    # Keep augmentation close to the original notebook.
    fliplr=0.5,
    flipud=0.5,
    degrees=12.0,
    translate=0.04,
    scale=0.08,
    shear=0.0,
    perspective=0.0,

    # Grayscale source images: no hue/saturation changes.
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.15,

    # Avoid synthetic scene composition for this baseline.
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    cache=False,
    plots=True,
    verbose=True,
)

training_time = time.perf_counter() - training_start

run_dir = Path(model.trainer.save_dir)
best_checkpoint = run_dir / "weights" / "best.pt"

if not best_checkpoint.is_file():
    raise FileNotFoundError(
        f"Best checkpoint was not created: {best_checkpoint}"
    )

model = YOLO(str(best_checkpoint))

print(
    f"Training complete | "
    f"run directory: {run_dir} | "
    f"best checkpoint: {best_checkpoint} | "
    f"time: {training_time / 60:.1f} min"
)



## 6. Read training history

Ultralytics writes `results.csv` inside the training run directory. The plots below use YOLO's segmentation loss and mask mAP rather than the U-Net BCE + Dice history.


In [ ]:

history_path = run_dir / "results.csv"

if not history_path.is_file():
    raise FileNotFoundError(f"YOLO training history not found: {history_path}")

history = pd.read_csv(history_path)
history.columns = [column.strip() for column in history.columns]

print("Available history columns:")
print(", ".join(history.columns))

required_columns = [
    "train/seg_loss",
    "val/seg_loss",
    "metrics/mAP50(M)",
    "metrics/mAP50-95(M)",
]

missing = [column for column in required_columns if column not in history.columns]

if missing:
    print(f"Warning: missing expected columns: {missing}")

epochs = np.arange(1, len(history) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

if "train/seg_loss" in history.columns:
    axes[0].plot(
        epochs,
        history["train/seg_loss"],
        marker="o",
        label="Train segmentation loss",
    )

if "val/seg_loss" in history.columns:
    axes[0].plot(
        epochs,
        history["val/seg_loss"],
        marker="o",
        label="Validation segmentation loss",
    )

axes[0].set(
    title="YOLO segmentation loss",
    xlabel="Epoch",
    ylabel="Loss",
)
axes[0].legend()
axes[0].grid(alpha=0.25)

if "metrics/mAP50(M)" in history.columns:
    axes[1].plot(
        epochs,
        history["metrics/mAP50(M)"],
        marker="o",
        label="Mask mAP50",
    )

if "metrics/mAP50-95(M)" in history.columns:
    axes[1].plot(
        epochs,
        history["metrics/mAP50-95(M)"],
        marker="o",
        label="Mask mAP50-95",
    )

axes[1].set(
    title="YOLO validation segmentation metrics",
    xlabel="Epoch",
    ylabel="Metric",
    ylim=(0, 1),
)
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

print(f"Best checkpoint: {best_checkpoint}")



## 7. Prediction helpers

YOLO already predicts individual instances. For threshold selection, we merge the selected instance masks into a single foreground mask, matching the original notebook's pixel-Dice evaluation.

`retina_masks=True` asks Ultralytics for masks aligned to the original image resolution, which makes native-resolution submission generation straightforward.


In [ ]:

def to_yolo_image(image):
    if image.ndim != 2:
        raise ValueError(f"Expected grayscale image, got shape {image.shape}")

    return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)


def predict_instances(image, min_confidence=0.15):
    source = to_yolo_image(image)

    results = model.predict(
        source=source,
        imgsz=IMAGE_SIZE,
        conf=min_confidence,
        iou=0.7,
        retina_masks=True,
        device=YOLO_DEVICE,
        verbose=False,
    )

    result = results[0]

    if result.masks is None or result.boxes is None:
        return []

    masks = result.masks.data.detach().float().cpu().numpy()
    confidences = result.boxes.conf.detach().float().cpu().numpy()

    native_height, native_width = image.shape

    instances = []

    for mask, confidence in zip(masks, confidences):
        if mask.shape != image.shape:
            mask = cv2.resize(
                mask,
                (native_width, native_height),
                interpolation=cv2.INTER_NEAREST,
            )

        instances.append(
            {
                "mask": mask >= 0.5,
                "confidence": float(confidence),
            }
        )

    return instances


def union_instances(instances, confidence_threshold):
    if not instances:
        return None

    shape = instances[0]["mask"].shape
    foreground = np.zeros(shape, dtype=bool)

    for instance in instances:
        if instance["confidence"] >= confidence_threshold:
            foreground |= instance["mask"]

    return foreground


def get_record_target(image_id, image_shape):
    target = np.zeros(image_shape, dtype=np.uint8)

    ann_ids = coco.getAnnIds(imgIds=[image_id])

    for annotation in coco.loadAnns(ann_ids):
        target |= coco.annToMask(annotation).astype(np.uint8)

    return target > 0



## 8. Select an instance-confidence threshold on validation

The original U-Net baseline selected a foreground pixel threshold.

YOLO naturally gives an **instance confidence** instead, so this section searches confidence thresholds and selects the one with the best global pixel Dice on the validation records.


In [ ]:

threshold_stats = {
    threshold: [0.0, 0.0]
    for threshold in CONFIDENCE_CANDIDATES
}

validation_examples = []

minimum_confidence = min(CONFIDENCE_CANDIDATES)

validation_start = time.perf_counter()

progress = tqdm(
    val_records.itertuples(index=False),
    total=len(val_records),
    desc="Validation inference",
    unit="record",
)

for row in progress:
    info = coco.loadImgs([row.image_id])[0]

    image = cv2.imread(
        str(TRAIN_IMAGE_DIR / info["file_name"]),
        cv2.IMREAD_GRAYSCALE,
    )

    if image is None:
        raise OSError(f"Could not read validation image: {info['file_name']}")

    target = get_record_target(row.image_id, image.shape)
    instances = predict_instances(
        image,
        min_confidence=minimum_confidence,
    )

    for threshold, stats in threshold_stats.items():
        prediction = union_instances(instances, threshold)

        if prediction is None:
            prediction = np.zeros_like(target, dtype=bool)

        intersection = np.logical_and(prediction, target).sum()

        stats[0] += float(intersection)
        stats[1] += float(prediction.sum() + target.sum())

    if len(validation_examples) < 4:
        validation_examples.append(
            {
                "image": image,
                "target": target,
                "instances": instances,
                "file_name": info["file_name"],
            }
        )

validation_time = time.perf_counter() - validation_start

threshold_scores = {
    threshold: (2.0 * intersection + 1e-6) / (denominator + 1e-6)
    for threshold, (intersection, denominator) in threshold_stats.items()
}

threshold_summary = pd.DataFrame(
    [
        {
            "confidence": threshold,
            "pixel_dice": score,
        }
        for threshold, score in threshold_scores.items()
    ]
).sort_values(
    "pixel_dice",
    ascending=False,
).reset_index(drop=True)

BEST_CONFIDENCE = float(
    threshold_summary.iloc[0]["confidence"]
)

print(threshold_summary.to_string(index=False))
print(f"Selected confidence threshold: {BEST_CONFIDENCE:.2f}")
print(f"Validation inference time: {validation_time:.1f}s")



## 9. Generate the competition submission

Each YOLO instance mask is treated as a filament candidate.

Connected-component cleanup is still applied to stay consistent with the original submission pipeline. The final mask components are encoded using COCO's column-major RLE format.


In [ ]:

def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(
        np.asfortranarray(binary_mask.astype(np.uint8))
    )
    return encoded["counts"].decode("utf-8")


def split_into_filaments(binary_mask, min_area=MIN_COMPONENT_AREA):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8),
        connectivity=8,
    )

    components = [
        labels == label
        for label in range(1, count)
        if stats[label, cv2.CC_STAT_AREA] >= min_area
    ]

    components.sort(
        key=lambda component: (
            -int(component.sum()),
            int(np.where(component)[0].min()),
            int(np.where(component)[1].min()),
        )
    )

    return components


def get_test_image_paths():
    paths = sorted(
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg"}
    )

    if not paths:
        raise FileNotFoundError(
            f"No test JPEGs found in {TEST_IMAGE_DIR}"
        )

    if len({path.stem for path in paths}) != len(paths):
        raise ValueError(
            "Test images contain duplicate filename stems."
        )

    return paths


def create_submission(output_path=SUBMISSION_PATH):
    rows = []

    paths = get_test_image_paths()
    start_time = time.perf_counter()

    print(
        f"Generating predictions for {len(paths)} test images "
        f"using confidence {BEST_CONFIDENCE:.2f}..."
    )

    for image_index, path in enumerate(paths, start=1):
        image = cv2.imread(
            str(path),
            cv2.IMREAD_GRAYSCALE,
        )

        if image is None:
            raise OSError(f"Could not read test image: {path}")

        instances = predict_instances(
            image,
            min_confidence=BEST_CONFIDENCE,
        )

        filament_counter = 0

        for instance in instances:
            if instance["confidence"] < BEST_CONFIDENCE:
                continue

            components = split_into_filaments(
                instance["mask"]
            )

            for component in components:
                filament_counter += 1

                rows.append(
                    {
                        "filament_id": (
                            f"{path.stem}_{filament_counter}"
                        ),
                        "segmentation_rle": mask_to_rle(
                            component
                        ),
                    }
                )

        if image_index % 25 == 0 or image_index == len(paths):
            elapsed = time.perf_counter() - start_time
            rate = image_index / elapsed if elapsed > 0 else 0.0
            remaining = (
                (len(paths) - image_index) / rate
                if rate > 0
                else 0.0
            )

            print(
                f"Predicted {image_index}/{len(paths)} images | "
                f"filaments: {len(rows):,} | "
                f"speed: {rate:.2f} img/s | "
                f"ETA: {remaining:.0f}s"
            )

    submission = pd.DataFrame(
        rows,
        columns=[
            "filament_id",
            "segmentation_rle",
        ],
    )

    if submission["filament_id"].duplicated().any():
        raise ValueError(
            "Generated submission contains duplicate filament IDs."
        )

    submission.to_csv(
        output_path,
        index=False,
    )

    elapsed = time.perf_counter() - start_time

    print(
        f"Submission complete | "
        f"{len(paths)} images | "
        f"{len(submission):,} filament rows | "
        f"{elapsed / 60:.1f} min | "
        f"saved to {output_path}"
    )

    return submission


if CREATE_SUBMISSION:
    submission_df = create_submission()
    display(submission_df.head())



## 10. Validation predictions and threshold visualization

YOLO does not produce the same pixel-probability map as the U-Net baseline, so the qualitative view uses:

1. input image
2. ground-truth mask
3. predicted instance union
4. predicted instance count / confidence summary


In [ ]:

fig, axis = plt.subplots(figsize=(7, 4.5))

axis.plot(
    threshold_summary["confidence"],
    threshold_summary["pixel_dice"],
    marker="o",
)

axis.axvline(
    BEST_CONFIDENCE,
    linestyle="--",
    label=f"Selected: {BEST_CONFIDENCE:.2f}",
)

axis.set(
    title="Validation confidence threshold selection",
    xlabel="YOLO instance confidence",
    ylabel="Pixel Dice",
    ylim=(0, 1),
)

axis.grid(alpha=0.25)
axis.legend()

fig.tight_layout()
plt.show()


for example in validation_examples:
    image = example["image"]
    target = example["target"]
    instances = example["instances"]

    prediction = union_instances(
        instances,
        BEST_CONFIDENCE,
    )

    if prediction is None:
        prediction = np.zeros_like(target, dtype=bool)

    selected_instances = [
        instance
        for instance in instances
        if instance["confidence"] >= BEST_CONFIDENCE
    ]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14, 4.5),
    )

    axes[0].imshow(
        image,
        cmap="gray",
    )
    axes[0].set_title("Input image")

    axes[1].imshow(
        target,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[1].set_title("Ground truth")

    axes[2].imshow(
        prediction,
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[2].set_title(
        f"YOLO prediction · "
        f"{len(selected_instances)} instance(s)"
    )

    for axis in axes:
        axis.axis("off")

    fig.suptitle(example["file_name"])
    fig.tight_layout()
    plt.show()

    confidence_values = [
        instance["confidence"]
        for instance in selected_instances
    ]

    if confidence_values:
        print(
            f"{example['file_name']}: "
            f"{len(confidence_values)} selected instance(s) | "
            f"confidence range "
            f"{min(confidence_values):.3f}–"
            f"{max(confidence_values):.3f}"
        )
    else:
        print(
            f"{example['file_name']}: "
            "no instances above the selected confidence."
        )



## 11. Display a labeled prediction on an unseen test image

Test ground truth is not used.


In [ ]:

def plot_labeled_prediction(image, instances, title):
    selected_instances = [
        instance
        for instance in instances
        if instance["confidence"] >= BEST_CONFIDENCE
    ]

    fig, axis = plt.subplots(figsize=(9, 9))
    axis.imshow(image, cmap="gray")

    filament_index = 0

    for instance in selected_instances:
        components = split_into_filaments(
            instance["mask"]
        )

        for component in components:
            filament_index += 1

            axis.contour(
                component.astype(np.uint8),
                levels=[0.5],
                linewidths=1.2,
            )

            moments = cv2.moments(
                component.astype(np.uint8)
            )

            if moments["m00"]:
                center_x = (
                    moments["m10"] / moments["m00"]
                )
                center_y = (
                    moments["m01"] / moments["m00"]
                )
            else:
                ys, xs = np.where(component)
                center_x = xs.mean()
                center_y = ys.mean()

            area = int(component.sum())

            axis.text(
                center_x,
                center_y,
                (
                    f"F{filament_index}\n"
                    f"{area:,} px\n"
                    f"conf {instance['confidence']:.2f}"
                ),
                fontsize=8,
                ha="center",
                va="center",
                bbox=dict(
                    boxstyle="round,pad=0.2",
                    facecolor="white",
                    alpha=0.75,
                ),
            )

    axis.set_title(
        f"{title} · "
        f"{filament_index} predicted filament(s)"
    )
    axis.axis("off")

    fig.tight_layout()
    plt.show()


test_paths = get_test_image_paths()

test_path = test_paths[0]

test_image = cv2.imread(
    str(test_path),
    cv2.IMREAD_GRAYSCALE,
)

if test_image is None:
    raise OSError(
        f"Could not read test image: {test_path}"
    )

test_instances = predict_instances(
    test_image,
    min_confidence=BEST_CONFIDENCE,
)

print(
    f"Displaying labeled prediction for test image: "
    f"{test_path.name}"
)

plot_labeled_prediction(
    test_image,
    test_instances,
    (
        f"Test prediction · {test_path.name} · "
        f"confidence {BEST_CONFIDENCE:.2f}"
    ),
)



## 12. Experiment summary

Key files produced by this notebook:

- `yolo_filament_dataset/` — temporary YOLO-format training dataset
- `yolo_runs/yolo26s_filament/weights/best.pt` — best YOLO checkpoint
- `solar_filament_yolo26s_submission.csv` — competition submission

The model is **YOLO26s-seg**, pretrained on COCO and fine-tuned for one segmentation class: `filament`.
